# 5. Diagnóstico dos resultados RAW/TANH

Esta análise lê o registro do experimento de dez milhões de ASTs e compara as duas saídas: skeleton e skeleton com coeficientes. O corpus de avaliação contém skeletons distintos entre si e do treino.


In [1]:
from pathlib import Path
import json,pandas as pd
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').exists())
RUN_DIR=ROOT/'experiments/canonicalization/outputs/raw_tanh'
status=json.loads((RUN_DIR/'status.json').read_text(encoding='utf8'))
print(status['corpus'])


{'sampled_asts': 10000000, 'stored_examples': 9856116, 'rows_by_split': {'train': 6997384, 'validation': 952965, 'test': 1905767}, 'distinct_skeletons_by_split': {'train': 6665377, 'validation': 952965, 'test': 1905767}, 'distinct_skeletons_total': 9524109, 'operator_counts_by_split': {'train': {'6': 6922533, '5': 74100, '4': 738, '3': 13}, 'validation': {'6': 943243, '5': 9628, '4': 93, '3': 1}, 'test': {'6': 1886184, '5': 19410, '4': 171, '3': 2}}, 'protection_selected_by_split': {'train': 3497775, 'validation': 476553, 'test': 953714}, 'tanh_inserted_by_split': {'train': 3390899, 'validation': 474587, 'test': 949725}, 'skeleton_overlap': 0, 'parquet_root': '/home/vm02/mateus_witeck/tsfmxai/experiments/canonicalization/models/outputs/raw_tanh/parquet'}


O registro contém contagens estruturais do corpus e a separação dos skeletons.

In [2]:
rows=[]
for name,result in status['results'].items():
    rows.append({'Modelo':name,**{key:result.get(key) for key in
        ('count','skeleton_exact','term_f1','joint_exact','best_epoch')}})
display(pd.DataFrame(rows))


,Modelo,count,skeleton_exact,term_f1,joint_exact,best_epoch
0,transformer_skeleton,1905767,0.979441,0.995608,NaN,10
1,transformer_joint,1905767,0.973169,0.995042,0.042048,10


A tabela destaca acerto estrutural, acerto conjunto e melhor época. A célula seguinte mede o que a cabeça de coeficientes custa na estrutura e quantas saídas inválidas cada variante produziu. Um estudo de falhas por operador e número de TANH exige as previsões individuais, que o registro não guarda.

In [3]:
skeleton=status['results']['transformer_skeleton']; joint=status['results']['transformer_joint']
n=skeleton['count']
rows={'skeleton exato':'skeleton_exact','saída válida':'valid_rate','F1 por termo':'term_f1'}
comparison=pd.DataFrame({label:{'skeleton':skeleton[key],'conjunto':joint[key],'diferença (pp)':100*(joint[key]-skeleton[key])}
                         for label,key in rows.items()}).T
display(comparison)
for name,result in (('skeleton',skeleton),('conjunto',joint)):
    invalid=round((1-result['valid_rate'])*n); wrong=round((1-result['skeleton_exact'])*n)
    print(f"{name}: {wrong:,} skeletons errados de {n:,}, dos quais {invalid:,} saídas inválidas")
print(f"o acerto conjunto ({joint['joint_exact']:.4f}) é {joint['joint_exact']/joint['skeleton_exact']:.1%} dos skeletons acertados pela variante conjunta")

,skeleton,conjunto,diferença (pp)
skeleton exato,0.979441,0.973169,-0.627149
saída válida,0.991709,0.986935,-0.477393
F1 por termo,0.995608,0.995042,-0.056648


skeleton: 39,181 skeletons errados de 1,905,767, dos quais 15,801 saídas inválidas
conjunto: 51,133 skeletons errados de 1,905,767, dos quais 24,899 saídas inválidas
o acerto conjunto (0.0420) é 4.3% dos skeletons acertados pela variante conjunta


## Registro do experimento

**Pergunta:** como as variantes se comparam em skeletons novos? **Dados:** execução de dez milhões de ASTs do notebook `06_train.ipynb`, lida de `experiments/canonicalization/outputs/raw_tanh/status.json`; os 1.905.767 skeletons de teste são distintos entre si e do treino. **Proveniência:** commit `6fb7621e3633cf4c5734ddacf24f244348c4f4e5` e hashes de `manifest.json`. **Métricas:** skeleton exato, F1 por termo, taxa de saída válida e acerto conjunto, com geração livre. **Resultados:** variante de skeleton, 0,9794 de skeleton exato (39.181 errados, 15.801 saídas inválidas); variante conjunta, 0,9732 (51.133 errados, 24.899 inválidas), ou seja 0,63 ponto percentual a menos; o acerto conjunto da variante conjunta é 0,0420, isto é, 4,3% dos skeletons que ela acertou também têm todos os coeficientes dentro de 1%. **Resumo científico:** a cabeça de coeficientes degrada levemente a estrutura e não resolve os coeficientes, que dependem de produtos e somas de vários parâmetros da AST; a estrutura canônica é aprendida muito melhor do que a aritmética dos coeficientes. **Reprodução:** `.venv/Scripts/python.exe -m jupyter nbconvert --to notebook --execute --inplace experiments/canonicalization/08_errors.ipynb`.